<a href="https://colab.research.google.com/github/sittana-afifi/AIMS-KTT-Hackathon---Challenge-S2.T1.3/blob/main/recommender_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def run_recommender():
    # 1. Load the project files
    try:
        catalog = pd.read_csv('catalog.csv')
        click_log = pd.read_csv('click_log.csv')
    except FileNotFoundError:
        print("Error: catalog.csv or click_log.csv not found. Run data_generator.py first.")
        return

    # 2. FIX POSITION BIAS: Inverse Position Weighting
    # We found that users intentionally click Rank 4.
    # We give those clicks 50% more weight than Rank 1 clicks.
    clicks = click_log[click_log['clicked'] == 1].copy()
    clicks['weight'] = clicks['rank_shown'].apply(lambda r: 1.5 if r >= 4 else 1.0)

    # Sum weights per SKU to get a "Fair Popularity" score
    pop_scores = clicks.groupby('sku')['weight'].sum().reset_index()
    pop_scores.rename(columns={'weight': 'pop_score'}, inplace=True)

    # Merge and Normalize
    catalog = catalog.merge(pop_scores, on='sku', how='left').fillna(0)
    if catalog['pop_score'].max() > 0:
        catalog['pop_score'] = catalog['pop_score'] / catalog['pop_score'].max()

    # 3. TEXT SIMILARITY: TF-IDF
    # Combine title and category for better matching
    catalog['metadata'] = catalog['title'] + " " + catalog['category'] + " " + catalog['description']
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(catalog['metadata'])

    # 4. THE LOCAL BOOST LOGIC (The "Made in Rwanda" Nudge)
    # Identify Rwandan products based on origin_district
    catalog['is_rwanda'] = catalog['origin_district'].apply(lambda x: 1 if x != 'Global' else 0)

    def get_top_recommendations(query, n=5):
        # Calculate text similarity
        query_vec = tfidf.transform([query])
        sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

        temp_df = catalog.copy()
        temp_df['sim_score'] = sim_scores

        # FINAL SCORING FORMULA:
        # Score = (Similarity + Weighted Popularity) * Local Multiplier
        temp_df['final_score'] = (temp_df['sim_score'] * 0.6) + (temp_df['pop_score'] * 0.4)

        # Apply 20% "Made in Rwanda" boost
        temp_df['final_score'] = temp_df.apply(
            lambda x: x['final_score'] * 1.2 if x['is_rwanda'] == 1 else x['final_score'],
            axis=1
        )

        return temp_df.sort_values(by='final_score', ascending=False).head(n)

    # 5. EXECUTION & DISPLAY
    search_query = "handmade basket"
    results = get_top_recommendations(search_query)

    print(f"\\n--- 🇷🇼 RECOMMENDER RESULTS FOR: '{search_query}' ---")
    print(results[['sku', 'title', 'origin_district', 'final_score']])
    print("\\nLogic Applied:")
    print("- Position Bias: Rank 4+ clicks weighted at 1.5x.")
    print("- Local Nudge: Rwandan products received a 20% multiplier.")

if __name__ == "__main__":
    run_recommender()

Error: catalog.csv or click_log.csv not found. Run data_generator.py first.
